# HRAF Misfortune Model Inference

This notebook loads a pre-trained model and runs inference on new passages.

## Model Performance (on test set)
- **F1 Micro: 0.7266**
- **F1 Macro: 0.6584**

## Labels
The model predicts 12 labels:
- **Event types**: Illness, Accident, Other
- **Causes**: Material_Physical, Spirits_Gods, Witchcraft_Sorcery, Rule_Violation_Taboo
- **Actions**: Physical_Material, Technical_Specialist, Divination, Shaman_Medium_Healer, Priest_High_Religion

In [1]:
# ============================================================================
# CELL 1: IMPORTS
# ============================================================================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
from pathlib import Path
from typing import Dict, List, Optional
from transformers import AutoTokenizer, AutoModel, AutoConfig, PreTrainedModel, PretrainedConfig
from transformers.modeling_outputs import SequenceClassifierOutput

# Set device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

Using device: mps


In [2]:
# ============================================================================
# CELL 2: MODEL ARCHITECTURE (Required for loading)
# ============================================================================

class ConfigurableHierarchicalConfig(PretrainedConfig):
    """Configuration for the model"""
    model_type = "configurable_hierarchical"

    def __init__(
        self,
        base_model="roberta-base",
        use_hierarchy=False,
        gated_hierarchy=False,
        gate_threshold=0.5,
        hidden_size=768,
        hierarchical_hidden_size=768,
        num_hidden_layers=3,
        dropout=0.15,
        attention_dropout=0.1,
        use_weighted_loss=True,
        use_focal_loss=True,
        focal_gamma=4.0,
        teacher_forcing_ratio=0.0,
        predict_main_labels=False,
        num_main_labels=0,
        num_event_labels=3,
        num_cause_labels=4,
        num_action_labels=5,
        total_labels=12,
        label_indices=None,
        label_names=None,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.base_model = base_model
        self.use_hierarchy = use_hierarchy
        self.gated_hierarchy = gated_hierarchy
        self.gate_threshold = gate_threshold
        self.hidden_size = hidden_size
        self.hierarchical_hidden_size = hierarchical_hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.dropout = dropout
        self.attention_dropout = attention_dropout
        self.use_weighted_loss = use_weighted_loss
        self.use_focal_loss = use_focal_loss
        self.focal_gamma = focal_gamma
        self.teacher_forcing_ratio = teacher_forcing_ratio
        self.predict_main_labels = predict_main_labels
        self.num_main_labels = num_main_labels
        self.num_event_labels = num_event_labels
        self.num_cause_labels = num_cause_labels
        self.num_action_labels = num_action_labels
        self.total_labels = total_labels
        self.label_indices = label_indices or {}
        self.label_names = label_names or []


class ConfigurableHierarchicalModel(PreTrainedModel):
    """The model architecture"""
    config_class = ConfigurableHierarchicalConfig
    base_model_prefix = "configurable_hierarchical"
    supports_gradient_checkpointing = True

    def __init__(self, config: ConfigurableHierarchicalConfig):
        super().__init__(config)
        self.config = config
        self.encoder = AutoModel.from_pretrained(config.base_model)

        if hasattr(config, 'attention_dropout') and config.attention_dropout > 0:
            self.encoder.config.attention_probs_dropout_prob = config.attention_dropout

        self.main_classifier = None
        hierarchical_input_size = config.hidden_size

        self.event_classifier = self._build_sublabel_classifier(
            hierarchical_input_size, config.num_event_labels,
            config.hierarchical_hidden_size, config.num_hidden_layers, config.dropout
        )
        self.cause_classifier = self._build_sublabel_classifier(
            hierarchical_input_size, config.num_cause_labels,
            config.hierarchical_hidden_size, config.num_hidden_layers, config.dropout
        )
        self.action_classifier = self._build_sublabel_classifier(
            hierarchical_input_size, config.num_action_labels,
            config.hierarchical_hidden_size, config.num_hidden_layers, config.dropout
        )

        self.use_hierarchy = config.use_hierarchy
        self.gated_hierarchy = config.gated_hierarchy
        self.gate_threshold = config.gate_threshold
        self.post_init()

    def _build_sublabel_classifier(self, input_size, output_size, hidden_size, num_layers, dropout):
        if output_size == 0:
            return None
        layers = []
        for i in range(num_layers):
            if i == 0:
                layers.append(nn.Linear(input_size, hidden_size))
            else:
                layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(hidden_size, output_size))
        return nn.Sequential(*layers)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        teacher_forcing=False,
        return_dict=None,
        **kwargs
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled_output = encoder_outputs.last_hidden_state[:, 0]

        hierarchical_input = pooled_output

        event_logits = self.event_classifier(hierarchical_input) if self.event_classifier else torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)
        cause_logits = self.cause_classifier(hierarchical_input) if self.cause_classifier else torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)
        action_logits = self.action_classifier(hierarchical_input) if self.action_classifier else torch.zeros(pooled_output.shape[0], 0).to(pooled_output.device)

        logits = torch.cat([event_logits, cause_logits, action_logits], dim=1)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.float())

        if not return_dict:
            output = (logits,) + encoder_outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=encoder_outputs.hidden_states,
            attentions=encoder_outputs.attentions,
        )

# Register model with HuggingFace
AutoConfig.register("configurable_hierarchical", ConfigurableHierarchicalConfig)
AutoModel.register(ConfigurableHierarchicalConfig, ConfigurableHierarchicalModel)

print("Model architecture registered!")

Model architecture registered!


In [ ]:
# ============================================================================
# CELL 3: CONFIGURATION - SET YOUR MODEL PATH HERE
# ============================================================================

# Path to the trained model
# Update this to point to your trained model directory
MODEL_PATH = "models/training_20251202_145349(best)/final_model"

# Alternative: use the best model from October training
# MODEL_PATH = "models/training_20251014_124910(best)/final_model"

# Label names (in order)
LABEL_NAMES = [
    "Illness",
    "Accident",
    "Other",
    "Material_Physical",
    "Spirits_Gods",
    "Witchcraft_Sorcery",
    "Rule_Violation_Taboo",
    "Physical_Material",
    "Technical_Specialist",
    "Divination",
    "Shaman_Medium_Healer",
    "Priest_High_Religion"
]

print(f"Model path: {MODEL_PATH}")
print(f"Labels: {len(LABEL_NAMES)}")

Model path: models/training_20251202_145349/final_model
Labels: 12


In [4]:
# ============================================================================
# CELL 4: LOAD MODEL
# ============================================================================

print("Loading model...")
model = AutoModel.from_pretrained(MODEL_PATH)
model = model.to(device)
model.eval()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Load training info (contains optimal thresholds)
training_info_path = Path(MODEL_PATH) / "training_info.json"
if training_info_path.exists():
    with open(training_info_path) as f:
        training_info = json.load(f)
    optimal_thresholds = training_info.get("optimal_thresholds", {})
    test_results = training_info.get("test_results", {})
    
    print(f"\nModel test performance:")
    print(f"  F1 Micro: {test_results.get('eval_f1_micro', 'N/A'):.4f}")
    print(f"  F1 Macro: {test_results.get('eval_f1_macro', 'N/A'):.4f}")
else:
    optimal_thresholds = {}
    print("Warning: training_info.json not found, using default thresholds")

print("\nModel loaded successfully!")

Loading model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading tokenizer...

Model test performance:
  F1 Micro: 0.7266
  F1 Macro: 0.6584

Model loaded successfully!


In [5]:
# ============================================================================
# CELL 5: INFERENCE FUNCTIONS
# ============================================================================

def predict_single(text: str, use_optimal_thresholds: bool = True) -> Dict:
    """
    Predict labels for a single passage.
    
    Args:
        text: Passage text
        use_optimal_thresholds: Use per-label optimal thresholds
        
    Returns:
        Dictionary with predictions and probabilities
    """
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
    
    # Apply thresholds
    results = {}
    predicted_labels = []
    
    for i, label in enumerate(LABEL_NAMES):
        prob = float(probs[i])
        
        if use_optimal_thresholds and label in optimal_thresholds:
            threshold = optimal_thresholds[label].get('threshold', 0.5)
        else:
            threshold = 0.5
        
        predicted = prob > threshold
        
        results[label] = {
            'probability': prob,
            'threshold': threshold,
            'predicted': predicted
        }
        
        if predicted:
            predicted_labels.append(label)
    
    return {
        'labels': results,
        'predicted_labels': predicted_labels
    }


def predict_batch(texts: List[str], use_optimal_thresholds: bool = True, batch_size: int = 16) -> List[Dict]:
    """
    Predict labels for multiple passages.
    
    Args:
        texts: List of passage texts
        use_optimal_thresholds: Use per-label optimal thresholds
        batch_size: Batch size for processing
        
    Returns:
        List of prediction dictionaries
    """
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
        
        # Process each result
        for j, prob_row in enumerate(probs):
            labels_result = {}
            predicted_labels = []
            
            for k, label in enumerate(LABEL_NAMES):
                prob = float(prob_row[k])
                
                if use_optimal_thresholds and label in optimal_thresholds:
                    threshold = optimal_thresholds[label].get('threshold', 0.5)
                else:
                    threshold = 0.5
                
                predicted = prob > threshold
                labels_result[label] = {
                    'probability': prob,
                    'threshold': threshold,
                    'predicted': predicted
                }
                
                if predicted:
                    predicted_labels.append(label)
            
            results.append({
                'text': batch_texts[j][:100] + '...' if len(batch_texts[j]) > 100 else batch_texts[j],
                'labels': labels_result,
                'predicted_labels': predicted_labels
            })
    
    return results


def print_prediction(result: Dict):
    """Pretty print a prediction result"""
    print("\nPredicted Labels:")
    if result['predicted_labels']:
        for label in result['predicted_labels']:
            info = result['labels'][label]
            print(f"  \u2713 {label}: {info['probability']:.3f}")
    else:
        print("  (no labels predicted above threshold)")
    
    print("\nAll Probabilities:")
    sorted_labels = sorted(result['labels'].items(), key=lambda x: x[1]['probability'], reverse=True)
    for label, info in sorted_labels:
        bar = '\u2588' * int(info['probability'] * 20) + '\u2591' * (20 - int(info['probability'] * 20))
        marker = '\u2713' if info['predicted'] else ' '
        print(f"  {marker} {label:25s} {bar} {info['probability']:.3f} (threshold: {info['threshold']:.2f})")

print("Inference functions defined!")

Inference functions defined!


In [6]:
# ============================================================================
# CELL 6: SINGLE PASSAGE INFERENCE
# ============================================================================

# Example passage - modify this to test different texts
test_passage = """
The village healer was called when the child fell ill with a high fever. 
The elders believed the sickness was caused by angering the forest spirits, 
and a ritual was performed to appease them.
"""

print("="*60)
print("SINGLE PASSAGE PREDICTION")
print("="*60)
print(f"\nPassage: {test_passage.strip()}")

result = predict_single(test_passage)
print_prediction(result)

SINGLE PASSAGE PREDICTION

Passage: The village healer was called when the child fell ill with a high fever. 
The elders believed the sickness was caused by angering the forest spirits, 
and a ritual was performed to appease them.

Predicted Labels:
  ✓ Illness: 0.759
  ✓ Spirits_Gods: 0.817
  ✓ Physical_Material: 0.534
  ✓ Technical_Specialist: 0.559
  ✓ Shaman_Medium_Healer: 0.669

All Probabilities:
  ✓ Spirits_Gods              ████████████████░░░░ 0.817 (threshold: 0.60)
  ✓ Illness                   ███████████████░░░░░ 0.759 (threshold: 0.55)
  ✓ Shaman_Medium_Healer      █████████████░░░░░░░ 0.669 (threshold: 0.50)
  ✓ Technical_Specialist      ███████████░░░░░░░░░ 0.559 (threshold: 0.50)
  ✓ Physical_Material         ██████████░░░░░░░░░░ 0.534 (threshold: 0.50)
    Divination                █████████░░░░░░░░░░░ 0.481 (threshold: 0.60)
    Other                     █████████░░░░░░░░░░░ 0.469 (threshold: 0.55)
    Material_Physical         ████████░░░░░░░░░░░░ 0.408 (threshold: 

In [7]:
# ============================================================================
# CELL 7: EVALUATE ON TEST SET (data NOT used in training)
# ============================================================================

# EXACT test set: 996 passages with same cleaning as training
# (removes passages <50 chars, duplicates, and no-label passages)
TEST_FILE = "data/inference.xlsx"

from sklearn.metrics import f1_score

print("="*60)
print("EVALUATING ON HELD-OUT TEST SET")
print("="*60)

if Path(TEST_FILE).exists():
    print(f"\nLoading test set from: {TEST_FILE}")
    df = pd.read_excel(TEST_FILE)
    print(f"Test passages: {len(df)}")
    
    # Get passages
    texts = df['Passage'].dropna().tolist()
    
    # Run predictions
    print("\nRunning predictions...")
    results = predict_batch(texts)
    
    # Add predictions to dataframe
    for label in LABEL_NAMES:
        df[f'pred_{label}'] = [r['labels'][label]['predicted'] for r in results[:len(df)]]
        df[f'prob_{label}'] = [r['labels'][label]['probability'] for r in results[:len(df)]]
    
    # Calculate F1 scores
    print("\n" + "="*60)
    print("RESULTS ON HELD-OUT TEST SET")
    print("="*60)
    
    all_true = []
    all_pred = []
    
    print("\nPer-label F1 scores:")
    for label in LABEL_NAMES:
        if label in df.columns:
            true_vals = df[label].fillna(0).astype(int).values
            pred_vals = df[f'pred_{label}'].astype(int).values
            f1 = f1_score(true_vals, pred_vals, zero_division=0)
            all_true.append(true_vals)
            all_pred.append(pred_vals)
            bar = '█' * int(f1 * 20) + '░' * (20 - int(f1 * 20))
            print(f"  {label:25s} {bar} {f1:.3f}")
    
    # Overall metrics
    if all_true:
        all_true = np.column_stack(all_true)
        all_pred = np.column_stack(all_pred)
        f1_micro = f1_score(all_true, all_pred, average='micro', zero_division=0)
        f1_macro = f1_score(all_true, all_pred, average='macro', zero_division=0)
        
        print(f"\n{'='*60}")
        print(f"OVERALL F1 MICRO: {f1_micro:.4f}")
        print(f"OVERALL F1 MACRO: {f1_macro:.4f}")
        print(f"{'='*60}")
        print(f"\nExpected F1 Micro: ~0.7266 (from training evaluation)")
else:
    print(f"Test file not found: {TEST_FILE}")
    print("Run the training notebook first or regenerate the test split.")

EVALUATING ON HELD-OUT TEST SET

Loading test set from: data/inference.xlsx
Test passages: 2086

Running predictions...

RESULTS ON HELD-OUT TEST SET

Per-label F1 scores:
  Illness                   ███████████████░░░░░ 0.788
  Accident                  ████░░░░░░░░░░░░░░░░ 0.244
  Other                     █████████████░░░░░░░ 0.669
  Material_Physical         ██████████░░░░░░░░░░ 0.539
  Spirits_Gods              ██████████░░░░░░░░░░ 0.546
  Witchcraft_Sorcery        █████░░░░░░░░░░░░░░░ 0.291
  Rule_Violation_Taboo      ████████░░░░░░░░░░░░ 0.418
  Physical_Material         █████████████░░░░░░░ 0.680
  Technical_Specialist      ███████░░░░░░░░░░░░░ 0.395
  Divination                ███░░░░░░░░░░░░░░░░░ 0.151
  Shaman_Medium_Healer      █████░░░░░░░░░░░░░░░ 0.292
  Priest_High_Religion      ██████░░░░░░░░░░░░░░ 0.326

OVERALL F1 MICRO: 0.5836
OVERALL F1 MACRO: 0.4450

Expected F1 Micro: ~0.7266 (from training evaluation)


In [8]:
# ============================================================================
# CELL 8: INTERACTIVE TESTING
# ============================================================================

# Test multiple passages
test_passages = [
    "The man broke his leg when he fell from a tree while hunting.",
    "The shaman performed divination to determine which spirit caused the illness.",
    "She violated the taboo by entering the sacred grove during her menstrual period.",
    "The priest prayed to the gods for healing and offered sacrifices at the temple.",
    "A witch was blamed for casting a curse that caused the crops to fail.",
]

print("="*60)
print("BATCH PREDICTION EXAMPLES")
print("="*60)

results = predict_batch(test_passages)

for i, result in enumerate(results):
    print(f"\n--- Passage {i+1} ---")
    print(f"Text: {test_passages[i]}")
    print(f"Predicted: {', '.join(result['predicted_labels']) if result['predicted_labels'] else '(none)'}")

BATCH PREDICTION EXAMPLES

--- Passage 1 ---
Text: The man broke his leg when he fell from a tree while hunting.
Predicted: Accident, Material_Physical

--- Passage 2 ---
Text: The shaman performed divination to determine which spirit caused the illness.
Predicted: Illness, Spirits_Gods, Physical_Material, Divination, Shaman_Medium_Healer

--- Passage 3 ---
Text: She violated the taboo by entering the sacred grove during her menstrual period.
Predicted: Rule_Violation_Taboo

--- Passage 4 ---
Text: The priest prayed to the gods for healing and offered sacrifices at the temple.
Predicted: Priest_High_Religion

--- Passage 5 ---
Text: A witch was blamed for casting a curse that caused the crops to fail.
Predicted: Other, Witchcraft_Sorcery


In [9]:
# ============================================================================
# CELL 9: COMPARE WITH GROUND TRUTH (if available)
# ============================================================================

def evaluate_predictions(df: pd.DataFrame, label_columns: List[str]) -> Dict:
    """
    Evaluate predictions against ground truth labels.
    
    Args:
        df: DataFrame with both ground truth and predicted columns
        label_columns: List of label column names
        
    Returns:
        Dictionary with evaluation metrics
    """
    from sklearn.metrics import f1_score, precision_score, recall_score
    
    results = {}
    all_true = []
    all_pred = []
    
    for label in label_columns:
        pred_col = f'pred_{label}'
        
        if label in df.columns and pred_col in df.columns:
            true_vals = df[label].fillna(0).astype(int).values
            pred_vals = df[pred_col].astype(int).values
            
            f1 = f1_score(true_vals, pred_vals, zero_division=0)
            precision = precision_score(true_vals, pred_vals, zero_division=0)
            recall = recall_score(true_vals, pred_vals, zero_division=0)
            
            results[label] = {
                'f1': f1,
                'precision': precision,
                'recall': recall
            }
            
            all_true.append(true_vals)
            all_pred.append(pred_vals)
    
    if all_true:
        all_true = np.column_stack(all_true)
        all_pred = np.column_stack(all_pred)
        
        results['overall'] = {
            'f1_micro': f1_score(all_true, all_pred, average='micro', zero_division=0),
            'f1_macro': f1_score(all_true, all_pred, average='macro', zero_division=0)
        }
    
    return results

print("Evaluation function defined!")
print("To evaluate: results = evaluate_predictions(df, LABEL_NAMES)")

Evaluation function defined!
To evaluate: results = evaluate_predictions(df, LABEL_NAMES)


In [10]:
# ============================================================================
# CELL 10: OPTIMAL THRESHOLDS REFERENCE
# ============================================================================

print("Optimal Thresholds (from training):")
print("="*50)

if optimal_thresholds:
    for label in LABEL_NAMES:
        if label in optimal_thresholds:
            info = optimal_thresholds[label]
            print(f"  {label:25s} threshold={info['threshold']:.2f}  F1={info.get('f1_at_threshold', 'N/A')}")
        else:
            print(f"  {label:25s} threshold=0.50 (default)")
else:
    print("  No optimal thresholds loaded - using default 0.5 for all labels")

Optimal Thresholds (from training):
  Illness                   threshold=0.55  F1=0.896551724137931
  Accident                  threshold=0.65  F1=0.6415094339622641
  Other                     threshold=0.55  F1=0.6740506329113924
  Material_Physical         threshold=0.50  F1=0.5287769784172662
  Spirits_Gods              threshold=0.60  F1=0.7801652892561983
  Witchcraft_Sorcery        threshold=0.65  F1=0.7596153846153846
  Rule_Violation_Taboo      threshold=0.60  F1=0.6788990825688074
  Physical_Material         threshold=0.50  F1=0.7726120033812341
  Technical_Specialist      threshold=0.50  F1=0.5829145728643216
  Divination                threshold=0.60  F1=0.5277777777777778
  Shaman_Medium_Healer      threshold=0.50  F1=0.6388059701492538
  Priest_High_Religion      threshold=0.50  F1=0.7037037037037037
